# Catalyst Center — SDK Tutorial

This notebook walks through using the official **`catalystcentersdk`** Python SDK to interact with Cisco Catalyst Center.  
We will cover the full workflow: SDK initialisation, querying devices, pagination, and caching — comparing the SDK approach to raw REST calls along the way.

## What We'll Cover
1. **Why the SDK?** — SDK vs raw `requests`
2. **Imports & Setup**
3. **SDK Initialisation** — automatic authentication
4. **Basic Device Query** — no pagination
5. **Inspect a Device Object** — SDK attribute access
6. **Pagination** — `offset` and `limit` with the SDK
7. **Auto-Pagination** — fetch all devices automatically
8. **Caching** — avoid repeated API calls
9. **Exercises** — try it yourself

---

## Step 1: Why Use the SDK?

Both approaches talk to the **same REST API**, but the SDK removes boilerplate:

| Task | Raw `requests` | `catalystcentersdk` |
|------|---------------|---------------------|
| Authentication | Manual POST → store token → add header to every call | **Automatic** |
| Token refresh | Manual | **Automatic** |
| JSON parsing | `response.json()["response"]` | `result.response` (attribute) |
| Error handling | Manual status checks | Built-in exceptions |
| SSL / retries | Manual | Built-in |
| Pagination | Manual | Still manual — but clean |

> **Key insight**: the SDK does **not** auto-paginate.  
> You still control `offset` and `limit` — but without managing tokens or headers.

## Step 2: Imports & Setup

In [1]:
import os
import time
import warnings

# Suppress SSL warnings for demo purposes
warnings.filterwarnings("ignore", message=".*urllib3.*", category=Warning)

from dotenv import load_dotenv
from catalystcentersdk import api   # <-- the only import needed for API calls

print("✅ Imports OK")

✅ Imports OK


### What each import does

- **`catalystcentersdk.api`** — the SDK entry point; a single `CatalystCenterAPI` object gives access to every API category (devices, sites, issues, tags, …)
- **`dotenv`** — loads credentials from `.env` so they never appear in code
- **`time`** — used later for the caching demo

## Step 3: Load Credentials & Initialise the SDK

With raw `requests` you would:
1. POST to `/dna/system/api/v1/auth/token` with Basic Auth
2. Extract the token
3. Add `{"X-Auth-Token": token}` to every subsequent request header

The SDK does all of that for you — just pass your credentials once.

In [2]:
load_dotenv()

BASE_URL = os.getenv("CATALYST_BASE_URL")
USERNAME = os.getenv("CATALYST_USERNAME")
PASSWORD = os.getenv("CATALYST_PASSWORD")
VERSION  = os.getenv("CATALYST_VERSION")

print(f"Base URL : {BASE_URL}")
print(f"Username : {USERNAME}")
print(f"Version  : {VERSION}")

Base URL : https://10.48.90.165
Username : admin
Version  : 2.3.7.9


In [3]:
# Initialise the SDK — authentication happens automatically here.
# The SDK fetches an X-Auth-Token and refreshes it before it expires.
catalyst = api.CatalystCenterAPI(
    username=USERNAME,
    password=PASSWORD,
    base_url=BASE_URL,
    version=VERSION,
    verify=False,   # Skip SSL cert verification (use True + a cert in production)
)

print("✅ SDK initialised — authenticated automatically")
print(f"   Connected to : {BASE_URL}")
print(f"   API version  : {VERSION}")

✅ SDK initialised — authenticated automatically
   Connected to : https://10.48.90.165
   API version  : 2.3.7.9


### SDK object layout

The `catalyst` object exposes every API category as an attribute:

```python
catalyst.devices    # network device operations
catalyst.sites      # site hierarchy
catalyst.issues     # assurance issues
catalyst.tag        # tag management
catalyst.clients    # client health
# ... and many more
```

You never touch tokens, headers, or session management directly.

## Step 4: Basic Device Query (No Pagination)

The SDK method is `catalyst.devices.get_device_list()`.  
When called **without** `offset` or `limit` the API applies its own defaults:

- **Default offset**: 1 (first record)
- **Default limit**: 500 (the API maximum)

> ⚠️ For networks with ≤ 500 devices this returns everything.  
> For larger networks, devices beyond 500 are **silently dropped** — you won't see an error!

In [4]:
# Basic query — no pagination, no filters
# The SDK returns up to 500 devices (API default limit).
result = catalyst.devices.get_device_list()

# SDK responses are MyDict objects — access fields as attributes
devices = result.response

print(f"Devices returned : {len(devices)}")
print(f"(If your network has > 500 devices, use pagination — see Step 6)")

Devices returned : 4
(If your network has > 500 devices, use pagination — see Step 6)


### Common filter parameters

You can narrow the query without pagination:

```python
# Filter by device family
catalyst.devices.get_device_list(family="Switches and Hubs")
catalyst.devices.get_device_list(family="Routers")
catalyst.devices.get_device_list(family="Wireless Controller")

# Filter by reachability
catalyst.devices.get_device_list(reachability_status="Reachable")
catalyst.devices.get_device_list(reachability_status="Unreachable")

# Filter by role
catalyst.devices.get_device_list(role="ACCESS")
catalyst.devices.get_device_list(role="CORE")

# Filter by software version
catalyst.devices.get_device_list(software_version="17.3.4a")

# Combine filters
catalyst.devices.get_device_list(family="Routers", reachability_status="Reachable")
```

## Step 5: Inspect a Device Object

The SDK wraps each device in a `MyDict` object — a dict you can read either as `device["hostname"]` **or** `device.hostname`.

In [5]:
if devices:
    sample = devices[0]
    print("Sample device (first 10 fields):\n")
    for i, (key, value) in enumerate(sample.items()):
        if i >= 10:
            print(f"  … and {len(sample) - 10} more fields")
            break
        print(f"  {key:<35} {value}")
else:
    print("No devices returned.")

Sample device (first 10 fields):

  description                         Cisco IOS Software [Dublin], Catalyst L3 Switch Software (CAT9K_IOSXE), Version 17.12.4, RELEASE SOFTWARE (fc3) Technical Support: http://www.cisco.com/techsupport Copyright (c) 1986-2024 by Cisco Systems, Inc. Compiled Tue 23-Jul-24 09:40 by mcpre netconf enabled
  type                                Cisco Catalyst 9500 Switch
  lastUpdateTime                      1782896601684
  macAddress                          70:18:a7:61:b3:00
  deviceSupportLevel                  Supported
  softwareType                        IOS-XE
  softwareVersion                     17.12.4
  serialNumber                        FCW2245F1LQ
  collectionInterval                  Global Default
  dnsResolvedManagementAddress        10.48.90.171
  … and 43 more fields


In [6]:
# Both access styles work identically
if devices:
    d = devices[0]
    print("Attribute access  :", d.hostname)
    print("Dict-key access   :", d["hostname"])
    print("Safe .get() access:", d.get("hostname", "N/A"))  # returns 'N/A' if missing

Attribute access  : CSS-DNA-POD2-BORDER1.bru-css-dna-pod2.cisco.com
Dict-key access   : CSS-DNA-POD2-BORDER1.bru-css-dna-pod2.cisco.com
Safe .get() access: CSS-DNA-POD2-BORDER1.bru-css-dna-pod2.cisco.com


### Key device fields

| Attribute | Description |
|-----------|-------------|
| `hostname` | Device hostname |
| `managementIpAddress` | Management IP |
| `family` | Device family (Switches and Hubs, Routers, …) |
| `type` | Specific model type |
| `platformId` | Hardware model (e.g. C9300-48UXM) |
| `softwareVersion` | IOS / IOS-XE version |
| `reachabilityStatus` | Reachable \| Unreachable \| PingReachable |
| `role` | ACCESS \| DISTRIBUTION \| CORE \| BORDER_ROUTER |
| `serialNumber` | Hardware serial |
| `upTime` | How long the device has been up |

## Step 6: Pagination with the SDK

### Why pagination is necessary

The API hard-caps responses at **500 records per call**.  
Without explicit pagination, devices beyond position 500 are silently omitted.

### The two parameters

| Parameter | SDK argument | Meaning | Default (if omitted) |
|-----------|-------------|---------|----------------------|
| Starting position | `offset` | 1-based index of the first record to return | 1 |
| Page size | `limit` | How many records to return (max 500) | 500 |

```
offset=1,  limit=10  →  devices  1–10  (page 1)
offset=11, limit=10  →  devices 11–20  (page 2)
offset=21, limit=10  →  devices 21–30  (page 3)
```

In [7]:
def get_devices_page(offset=1, limit=10, **filters):
    """Fetch a single page of network devices via the SDK.

    Args:
        offset  (int): 1-based starting position (default 1).
        limit   (int): Max records to return — up to 500 (default 10).
        **filters    : Any SDK filter kwargs, e.g. family='Switches and Hubs'.

    Returns:
        list: SDK MyDict objects for this page (empty list on last page).
    """
    response = catalyst.devices.get_device_list(
        offset=offset,
        limit=limit,
        **filters,
    )
    return response.response or []


def print_devices(devices, title="Network Devices"):
    """Print SDK device objects as a formatted table."""
    print(f"\n{title}")
    print("=" * 100)
    print(f"{'HOSTNAME':<40} {'IP ADDRESS':<20} {'FAMILY':<25} {'SW VERSION'}")
    print("-" * 100)
    for d in devices:
        print(
            f"{getattr(d, 'hostname', 'N/A'):<40}"
            f" {getattr(d, 'managementIpAddress', 'N/A'):<20}"
            f" {getattr(d, 'family', 'N/A'):<25}"
            f" {getattr(d, 'softwareVersion', 'N/A')}"
        )
    print("-" * 100)
    print(f"Total: {len(devices)} device(s)\n")


print("✅ Helper functions defined")

✅ Helper functions defined


### Page 1: first 10 devices

In [8]:
page1 = get_devices_page(offset=1, limit=10)
print_devices(page1, "Page 1 — devices 1–10")
print(f"Received {len(page1)} device(s)")


Page 1 — devices 1–10
HOSTNAME                                 IP ADDRESS           FAMILY                    SW VERSION
----------------------------------------------------------------------------------------------------
CSS-DNA-POD2-BORDER1.bru-css-dna-pod2.cisco.com 10.48.90.171         Switches and Hubs         17.12.4
CSS-DNA-POD2-EDGE2.bru-css-pod2.cisco.com 10.48.90.174         Switches and Hubs         17.12.6
CSS-DNA-POD2-WLC1.bru-css-pod2.cisco.com 10.48.90.178         Wireless Controller       17.15.20250606:191825
POD2-9800-1.bru-css-pod1-cisco.com       10.48.90.179         Wireless Controller       17.15.4d
----------------------------------------------------------------------------------------------------
Total: 4 device(s)

Received 4 device(s)


### Page 2: next 10 devices

Increment `offset` by the page size (`limit`) to move to the next page:

```
Page N  →  offset = (N - 1) * limit + 1
Page 2  →  offset = (2 - 1) * 10 + 1 = 11
Page 3  →  offset = (3 - 1) * 10 + 1 = 21
```

In [9]:
page2 = get_devices_page(offset=11, limit=10)
print_devices(page2, "Page 2 — devices 11–20")
print(f"Received {len(page2)} device(s)")
if len(page2) < 10:
    print("💡 Got fewer than 10 → this is the last page")


Page 2 — devices 11–20
HOSTNAME                                 IP ADDRESS           FAMILY                    SW VERSION
----------------------------------------------------------------------------------------------------
----------------------------------------------------------------------------------------------------
Total: 0 device(s)

Received 0 device(s)
💡 Got fewer than 10 → this is the last page


### Filtered page: first 25 switches

Pagination and filters can be combined freely.

In [10]:
switches = get_devices_page(
    offset=1,
    limit=25,
    family="Switches and Hubs",
)
print_devices(switches, "First 25 Switches")


First 25 Switches
HOSTNAME                                 IP ADDRESS           FAMILY                    SW VERSION
----------------------------------------------------------------------------------------------------
CSS-DNA-POD2-BORDER1.bru-css-dna-pod2.cisco.com 10.48.90.171         Switches and Hubs         17.12.4
CSS-DNA-POD2-EDGE2.bru-css-pod2.cisco.com 10.48.90.174         Switches and Hubs         17.12.6
----------------------------------------------------------------------------------------------------
Total: 2 device(s)



## Step 7: Auto-Pagination — Fetch ALL Devices

The SDK does **not** loop automatically.  
We keep advancing `offset` until the API returns an empty list or a partial page (fewer records than `limit`) — that signals the last page.

```
Call 1: offset=1,   limit=50  →  50 devices   (full page → continue)
Call 2: offset=51,  limit=50  →  50 devices   (full page → continue)
Call 3: offset=101, limit=50  →  25 devices   (partial  → STOP)

Total: 125 devices, 3 API calls
```

In [11]:
def get_all_devices(page_size=50, **filters):
    """Retrieve ALL devices by automatically iterating through pages.

    The SDK exposes get_device_list(offset, limit) but does not loop
    automatically.  This function handles the loop, stopping when:
      • the API returns an empty list, OR
      • the page is smaller than page_size (last page reached).

    Args:
        page_size (int): Records per SDK call.  Default 50, max 500.
                         Larger → fewer calls but bigger payloads.
        **filters       : Passed to get_device_list() on every page, e.g.
                          family='Routers', reachability_status='Reachable'.

    Returns:
        list: All matching SDK device objects.
    """
    all_devices = []
    offset = 1

    while True:
        print(f"  SDK call → offset={offset:>5}, limit={page_size} ...")
        page = get_devices_page(offset=offset, limit=page_size, **filters)

        if not page:
            break                        # empty response → done

        all_devices.extend(page)
        print(f"  ↳ {len(page):>3} device(s) received  (running total: {len(all_devices)})")

        if len(page) < page_size:
            break                        # partial page → last page

        offset += page_size              # advance to next page

    return all_devices


print("🔄 Fetching ALL devices (page_size=50) ...\n")
all_devices = get_all_devices(page_size=50)
print(f"\n✅ Done — total devices: {len(all_devices)}")
print_devices(all_devices[:5], f"First 5 of {len(all_devices)} total devices")

🔄 Fetching ALL devices (page_size=50) ...

  SDK call → offset=    1, limit=50 ...
  ↳   4 device(s) received  (running total: 4)

✅ Done — total devices: 4

First 5 of 4 total devices
HOSTNAME                                 IP ADDRESS           FAMILY                    SW VERSION
----------------------------------------------------------------------------------------------------
CSS-DNA-POD2-BORDER1.bru-css-dna-pod2.cisco.com 10.48.90.171         Switches and Hubs         17.12.4
CSS-DNA-POD2-EDGE2.bru-css-pod2.cisco.com 10.48.90.174         Switches and Hubs         17.12.6
CSS-DNA-POD2-WLC1.bru-css-pod2.cisco.com 10.48.90.178         Wireless Controller       17.15.20250606:191825
POD2-9800-1.bru-css-pod1-cisco.com       10.48.90.179         Wireless Controller       17.15.4d
----------------------------------------------------------------------------------------------------
Total: 4 device(s)



### Choosing the right page_size

| page_size | API calls for 500 devices | API calls for 5 000 devices | Trade-off |
|-----------|--------------------------|----------------------------|-----------|
| 10 | 50 | 500 | Many small calls |
| **50** | **10** | **100** | **Good balance** |
| 500 | 1 | 10 | Fewer calls, large payloads |

For most environments `page_size=50` or `page_size=100` is a reasonable choice.

## Step 8: Caching — Avoid Repeating Expensive Calls

Every `get_all_devices()` call makes multiple API round-trips.  
A simple **in-memory cache** stores the result for 5 minutes so repeated calls hit no API at all.

```
t=  0s  →  Cache MISS  → makes N SDK calls  → stores result
t= 30s  →  Cache HIT   → returns instantly, 0 SDK calls  ⚡
t=290s  →  Cache HIT   → still valid (< 300 s)
t=310s  →  Cache EXPIRED → fetches fresh data
```

### Impact

| Scenario | Without cache | With 5-min cache |
|----------|--------------|------------------|
| Dashboard refreshes every 30 s | **120 API calls / hour** | **12 API calls / hour** |
| Savings | — | **90 % reduction** |

In [12]:
_cache    = {}
CACHE_TTL = 300   # 5 minutes


def get_all_devices_cached(page_size=50, cache_ttl=CACHE_TTL, **filters):
    """Cached wrapper for get_all_devices().

    Uses a separate cache entry per unique filter combination, so
    caching switches does not pollute the cache for routers.

    Args:
        page_size (int): Passed to get_all_devices() on a cache miss.
        cache_ttl (int): Seconds before the cached entry expires (default 300).
        **filters       : Passed to get_all_devices() on a cache miss.

    Returns:
        dict with keys:
            devices   (list) – device objects
            cached    (bool) – True if no API call was made
            cache_age (float) – seconds since last cache population (0 = fresh)
    """
    cache_key    = f"devices_{hash(str(sorted(filters.items())))}"
    current_time = time.time()

    if cache_key in _cache:
        age = current_time - _cache[cache_key]["timestamp"]
        if age < cache_ttl:
            print(f"✅ Cache HIT  — age: {age:.1f}s  (TTL: {cache_ttl}s)  ⚡ No SDK calls!")
            return {"devices": _cache[cache_key]["data"], "cached": True, "cache_age": age}
        print(f"⏰ Cache EXPIRED — age {age:.1f}s > TTL {cache_ttl}s")
    else:
        print("❌ Cache MISS — fetching via SDK...")

    devices = get_all_devices(page_size=page_size, **filters)
    _cache[cache_key] = {"data": devices, "timestamp": time.time()}
    print(f"💾 Cached {len(devices)} device(s) for {cache_ttl}s")

    return {"devices": devices, "cached": False, "cache_age": 0}


print("✅ Cache helpers defined")

✅ Cache helpers defined


In [13]:
# --- Demo: two calls, only one hits the API ---

print("=" * 60)
print("CALL 1 (t=0s) — expect MISS")
print("=" * 60)
r1 = get_all_devices_cached(page_size=50)
print(f"Devices: {len(r1['devices'])}  |  From cache: {r1['cached']}\n")

time.sleep(2)

print("=" * 60)
print("CALL 2 (t≈2s) — expect HIT")
print("=" * 60)
r2 = get_all_devices_cached(page_size=50)
print(f"Devices: {len(r2['devices'])}  |  From cache: {r2['cached']}  |  Age: {r2['cache_age']:.1f}s")
print(f"\n⏰ Cache expires in ≈ {CACHE_TTL - r2['cache_age']:.0f}s")

CALL 1 (t=0s) — expect MISS
❌ Cache MISS — fetching via SDK...
  SDK call → offset=    1, limit=50 ...
  ↳   4 device(s) received  (running total: 4)
💾 Cached 4 device(s) for 300s
Devices: 4  |  From cache: False

CALL 2 (t≈2s) — expect HIT
✅ Cache HIT  — age: 2.0s  (TTL: 300s)  ⚡ No SDK calls!
Devices: 4  |  From cache: True  |  Age: 2.0s

⏰ Cache expires in ≈ 298s


---

## Complete Workflow — All in One

Everything from initialisation to a cached paginated result in one cell:

In [14]:
# 1. SDK is already initialised as `catalyst` above

# 2. Fetch all devices with auto-pagination + caching
result = get_all_devices_cached(page_size=50)

# 3. Display
print_devices(result["devices"], f"All Devices (cached={result['cached']})")

✅ Cache HIT  — age: 23.3s  (TTL: 300s)  ⚡ No SDK calls!

All Devices (cached=True)
HOSTNAME                                 IP ADDRESS           FAMILY                    SW VERSION
----------------------------------------------------------------------------------------------------
CSS-DNA-POD2-BORDER1.bru-css-dna-pod2.cisco.com 10.48.90.171         Switches and Hubs         17.12.4
CSS-DNA-POD2-EDGE2.bru-css-pod2.cisco.com 10.48.90.174         Switches and Hubs         17.12.6
CSS-DNA-POD2-WLC1.bru-css-pod2.cisco.com 10.48.90.178         Wireless Controller       17.15.20250606:191825
POD2-9800-1.bru-css-pod1-cisco.com       10.48.90.179         Wireless Controller       17.15.4d
----------------------------------------------------------------------------------------------------
Total: 4 device(s)



---

## Step 9: Exercises — Try It Yourself

### Exercise 1: Query Only Routers

In [15]:
# Fetch all routers (uncomment and run)
# routers = get_all_devices(page_size=50, family="Routers")
# print_devices(routers, f"All Routers ({len(routers)} total)")

### Exercise 2: Only Unreachable Devices

In [16]:
# Find devices that are currently unreachable
# unreachable = get_all_devices(page_size=50, reachability_status="Unreachable")
# print_devices(unreachable, f"Unreachable Devices ({len(unreachable)} total)")

### Exercise 3: Count Devices Without Fetching Them

For large networks, get the count first to plan pagination:

In [ ]:
# The SDK also exposes a count endpoint — much cheaper than fetching all devices
count_result = catalyst.devices.get_device_count()
total = count_result.response
print(f"Total devices in Catalyst Center : {total}")

page_size  = 50
pages_needed = -(-total // page_size)   # ceiling division
print(f"Pages needed with page_size={page_size} : {pages_needed}")

### Exercise 4: Display Extra Fields

In [ ]:
# Richer table with more fields
def print_detailed(devices):
    for d in devices:
        print(f"\n{'─'*60}")
        print(f"  Hostname  : {d.hostname}")
        print(f"  IP        : {d.managementIpAddress}")
        print(f"  Model     : {d.platformId}")
        print(f"  SW        : {d.softwareVersion}")
        print(f"  Role      : {d.role}")
        print(f"  Status    : {d.reachabilityStatus}")
        print(f"  Uptime    : {d.upTime}")

# Uncomment to try on the first page:
# first_page = get_devices_page(offset=1, limit=5)
# print_detailed(first_page)

---

## Additional SDK Resources

### Other SDK categories you can explore:

```python
# Sites
catalyst.sites.get_site()

# Assurance issues
catalyst.issues.get_issues()

# Client health
catalyst.clients.get_client_detail(mac_address="aa:bb:cc:dd:ee:ff")

# Tags
catalyst.tag.get_tag(sort_by='name', order='asc')

# Interface details
catalyst.devices.get_interface_info_by_id(device_id="<uuid>")

# Device config
catalyst.devices.get_device_config_by_id(network_device_id="<uuid>")
```

### Official links
- [catalystcentersdk on PyPI](https://pypi.org/project/catalystcentersdk/)
- [SDK GitHub repository](https://github.com/cisco-en-programmability/catalystcentersdk)
- [Catalyst Center API docs](https://developer.cisco.com/docs/dna-center/)